# Object Tracking Project — Football Player Tracking
## Part 4: Unified Detection and Tracking with Ultralytics (YOLO)

In the previous notebooks, we trained a detector with SSD (mAP ≈ 42 on test) and implemented tracking with the BoxMOT library (kept separate from the detector).

In this notebook we use a much simpler approach: the **Ultralytics** library. Since the project dataset was already prepared in YOLO format (including a `data.yaml` file), we can:

1. Fine-tune a **YOLO** model directly on this dataset (no annotation conversion needed, unlike what we did for SSD).
2. Use the `model.track(...)` method that Ultralytics provides out of the box; in this case we don't need to manage the detector and tracker separately — we just specify which tracker algorithm to use (e.g. `bytetrack.yaml` or `botsort.yaml`), and the rest of the pipeline (running the detector on each frame, format conversion, calling the tracker, drawing the result) is handled automatically.

This notebook has two parts: **Fine Tune** (training YOLO on the dataset) and **Tracking** (running tracking with the fine-tuned model).


### Importing the YOLO Class from Ultralytics


In [1]:
from ultralytics import YOLO

# Fine Tune

In this section we fine-tune a YOLO model on the football players dataset.


### Loading the Base YOLO Model

We use the lightweight `yolo11n.pt` (Nano) checkpoint. Heavier YOLO variants could be used for higher accuracy, but the Nano version gives us faster execution for this project.


In [2]:
model = YOLO("yolo11n.pt")

### Training (Fine-Tuning) on the Dataset

We just need to pass the path to `data.yaml`, which describes the dataset (paths and classes), to the `train` function. The model is automatically saved under `runs/detect/train/weights/` (including `best.pt` and `last.pt`).


In [3]:
result = model.train(data='football_players_detection/data.yaml',
                     epochs=10,
                     imgsz=320,
                     batch=128,
                     device=0)

New https://pypi.org/project/ultralytics/8.3.186 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.174 🚀 Python-3.10.12 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=football_players_detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=

train: Scanning /mnt/d/course record/ComputerVision/file & codes/Object Tracking/football_players_detection/train/labels


val: Fast image access ✅ (ping: 2.2±0.6 ms, read: 33.4±7.1 MB/s, size: 158.4 KB)


val: Scanning /mnt/d/course record/ComputerVision/file & codes/Object Tracking/football_players_detection/valid/labels.c


Plotting labels to runs/detect/train2/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001667, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.001), 87 bias(decay=0.0)
Image sizes 320 train, 320 val
Using 8 dataloader workers
Logging results to runs/detect/train2
Starting training for 10 epochs...
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      10/10      4.22G      1.191     0.6314     0.9374        294        320: 100%|██████████| 81/81 [00:33<00:00,  2.4
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:07<00

                   all        972       5407      0.888      0.547      0.603      0.378



10 epochs completed in 0.107 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 5.4MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 5.4MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.174 🚀 Python-3.10.12 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:07<00


                   all        972       5407      0.888      0.547      0.604      0.377
              football        241        275      0.863      0.206      0.285      0.147
                player        936       5132      0.914      0.888      0.923      0.608
Speed: 0.1ms preprocess, 0.6ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/train2


### Loading the Trained Best Model

From the saved weights, we load `best.pt` (the best model according to validation).


In [4]:
best_model = YOLO('runs/detect/train/weights/best.pt')

### Evaluation on the Test Set (mAP)

Similar to what we did for SSD, we evaluate the model on the test set.


In [5]:
metrics_val = best_model.val(data='./football_players_detection/data.yaml', split='test')

Ultralytics 8.3.174 🚀 Python-3.10.12 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.1 ms, read: 53.6±22.6 MB/s, size: 119.3 KB)


val: Scanning /mnt/d/course record/ComputerVision/file & codes/Object Tracking/football_players_detection/test/labels.ca
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 33/33 [00:05<


                   all        520       2903      0.787      0.515      0.564      0.325
              football        127        145       0.67      0.168      0.228     0.0983
                player        494       2758      0.905      0.861        0.9      0.552
Speed: 0.1ms preprocess, 3.5ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs/detect/val4


Displaying the evaluation results (including mAP) as a dictionary.

The result of this evaluation was **mAP ≈ 56** on the test set; compared to SSD (which reached mAP ≈ 42), this is roughly a 14–15 point improvement just from switching the detector from SSD to YOLO — with no change to the dataset. This shows how much detector quality affects the final tracking quality: **good tracking cannot achieve good results without good detection.**


In [6]:
metrics_val.results_dict

{'metrics/precision(B)': 0.7870358507807558,
 'metrics/recall(B)': 0.5145963072381721,
 'metrics/mAP50(B)': 0.5638811374787921,
 'metrics/mAP50-95(B)': 0.3248935628405239,
 'fitness': 0.34879232030435076}

### Evaluation on the Train Set

To check the gap between train and test performance (i.e. any potential overfitting), we repeat evaluation on the training set as well.


In [7]:
metrics_train = best_model.val(data='./football_players_detection/data.yaml', split='train')

Ultralytics 8.3.174 🚀 Python-3.10.12 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
val: Fast image access ✅ (ping: 1.1±0.4 ms, read: 49.7±15.1 MB/s, size: 102.1 KB)


val: Scanning /mnt/d/course record/ComputerVision/file & codes/Object Tracking/football_players_detection/train/labels.c
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 645/645 [01:0


                   all      10308      48810      0.851      0.548      0.604      0.353
              football       2107       2325      0.829      0.212      0.289      0.137
                player       9616      46485      0.874      0.884      0.918      0.569
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs/detect/val5


Displaying the evaluation results on the training set.


In [8]:
metrics_train.results_dict

{'metrics/precision(B)': 0.8513070136337297,
 'metrics/recall(B)': 0.5481372199729773,
 'metrics/mAP50(B)': 0.6037147671151157,
 'metrics/mAP50-95(B)': 0.3530603417491317,
 'fitness': 0.37812578428573007}

# Tracking

Now that the detector (YOLO) has been trained and evaluated, we use this model's `track` method to run tracking on the video directly.


### Running Tracking with a Single Method Call

Calling `best_model.track(...)`:

* `source`: path to the input video
* `save=False` / `show=True`: the tracking result is displayed live but not saved (if `save=True`, the output video with bounding boxes and IDs would be saved)
* `tracker="bytetrack.yaml"`: choice of tracking algorithm. In Ultralytics, `"botsort.yaml"` can be used instead (the same two algorithms we implemented manually with BoxMOT in the previous notebooks)

Worth noting: the entire pipeline (running the detector on each frame, converting its output to the format the tracker expects, calling the tracker, and drawing the result on the frame) is handled automatically by Ultralytics — exactly what we did manually (using BoxMOT) in notebooks `03` and `04`.

---

## Final Project Summary

The final pipeline of this project (tracking by detection) looks like this:

```text
Video → Frame → Object Detector (YOLO) → Bounding Box + Confidence
                                                    │
                                                    ▼
                                    Tracker (ByteTrack / BoT-SORT)
                                                    │
                                                    ▼
                                          Persistent Player IDs
                                                    │
                                                    ▼
                                            Tracked Video
```

Key takeaways from this project (across the four notebooks):

* In the tracking-by-detection approach, training the detector requires no video; a plain image dataset (image + bounding box + class) is enough.
* Detector quality directly affects final tracking quality (YOLO with mAP ≈ 56 outperformed SSD's mAP ≈ 42).
* Among tracking algorithms, **ByteTrack** is better suited for speed and real-time use, while **BoT-SORT** (using Re-ID/appearance features) is better suited when higher re-identification accuracy and fewer ID switches matter more.
* Using **Ultralytics** for pipelines whose detector is YOLO makes the process much simpler and faster, without needing to manage the detector and tracker separately.


In [11]:
results = best_model.track(source="football_players_detection/video/video2.mp4",
                           save=False,
                           show=True,
                           tracker="bytetrack.yaml")


WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/807) /mnt/d/course record/ComputerVision/file & codes/Object Tracking/football_players_detection/video/video2.mp4: 192x320 8 players, 13.9ms
video 1/1 (frame 2/807) /mnt/d/course record/ComputerVision/file & codes/Object Tracking/football_players_detection/video/video2.mp4: 192x320 8 players, 11.1ms
video 1/1 (frame 3/807) /mnt/d/course record/ComputerVision/file & codes/Object Tracking/football_players_detection/video/video2.mp4: 19